# EXP_08 (Phase 2): Resume Training to Break >99.00% Accuracy (Ultra-Low VRAM Footprint)

**Objective:** Continue fine-tuning `ConvNeXt-Small` from Epoch 10 checkpoint (`98.82% Val / 98.62% Test`) for **10 additional epochs (Epochs 11 to 20)** using micro-fine-tuning LR (`4e-5`), EMA (`0.9999`), Mixup/CutMix, and AMP FP16 to reach **🏆 >99.00% Accuracy**.

In [1]:
# Cell 1: Setup & Aggressive GPU VRAM Cleanup
import os
import sys
import time
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import numpy as np

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.dataloader import get_cifar10_loaders
from src.eval.evaluate_model import CIFAR10_CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] PyTorch Version: {torch.__version__}")
print(f"[Setup] Device: {device}")
if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / (1024**3)
    print(f"[Setup] GPU Memory: {free_mem:.2f} GB Free / {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB Total")


[Setup] PyTorch Version: 2.13.0+cu130
[Setup] Device: cuda
[Setup] GPU Memory: 3.57 GB Free / 3.68 GB Total


In [2]:
# Cell 2: Data Loaders (Batch Size 8 + Grad Accum 4 for Ultra-Low VRAM <1GB)
class MixupCutMixCollator:
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.8, num_classes=10):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.num_classes = num_classes

    def rand_bbox(self, size, lam):
        W, H = size[2], size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
        cx, cy = np.random.randint(W), np.random.randint(H)
        bbx1, bby1 = np.clip(cx - cut_w // 2, 0, W), np.clip(cy - cut_h // 2, 0, H)
        bbx2, bby2 = np.clip(cx + cut_w // 2, 0, W), np.clip(cy + cut_h // 2, 0, H)
        return bbx1, bby1, bbx2, bby2

    def __call__(self, batch):
        images, labels = torch.utils.data.dataloader.default_collate(batch)
        if np.random.rand() > self.prob:
            return images, F.one_hot(labels, self.num_classes).float()

        one_hot_labels = F.one_hot(labels, self.num_classes).float()
        if np.random.rand() > 0.5 and self.cutmix_alpha > 0:
            lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
            rand_index = torch.randperm(images.size(0))
            bbx1, bby1, bbx2, bby2 = self.rand_bbox(images.size(), lam)
            images[:, :, bbx1:bbx2, bby1:bby2] = images[rand_index, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size(2) * images.size(3)))
            targets = one_hot_labels * lam + one_hot_labels[rand_index] * (1. - lam)
        else:
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            rand_index = torch.randperm(images.size(0))
            images = lam * images + (1 - lam) * images[rand_index]
            targets = lam * one_hot_labels + (1 - lam) * one_hot_labels[rand_index]
        return images, targets

IMAGE_SIZE = 224
BATCH_SIZE = 8  # Reduced to Batch Size 8 for ultra-low VRAM footprint
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.RandAugment(num_ops=2, magnitude=12),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.25, scale=(0.02, 0.2))
])
eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mixup_collator = MixupCutMixCollator(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.8)
train_loader, val_loader, test_loader = get_cifar10_loaders(
    train_transform=train_transform, eval_transform=eval_transform,
    batch_size=BATCH_SIZE, num_workers=0
)
print(f"[Data] Configured: Batch Size={BATCH_SIZE} (Ultra-low VRAM footprint).")


[Data] Configured: Batch Size=8 (Ultra-low VRAM footprint).


In [3]:
# Cell 3: Model Builder & EMA Class
def build_model_sota(model_name="convnext_small", num_classes=10, device=device):
    from torchvision.models import convnext_small, ConvNeXt_Small_Weights
    model = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, num_classes)
    return model.to(device)

class ExponentialMovingAverage:
    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

print("[Model] Builder and EMA defined.")


[Model] Builder and EMA defined.


In [4]:
# Cell 4: Load Epoch 10 Checkpoint & Memory Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

checkpoint_dir = os.path.join(PROJECT_ROOT, "experiments", "checkpoints")
prev_ckpt_path = os.path.join(checkpoint_dir, "exp08_convnext_small_sota_best.pt")
new_ckpt_path = os.path.join(checkpoint_dir, "exp08_convnext_small_20epochs_peak.pt")

model = build_model_sota("convnext_small", num_classes=10, device=device)
if os.path.exists(prev_ckpt_path):
    state_dict = torch.load(prev_ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    print(f"[Resume] Successfully loaded Epoch 10 Checkpoint from {prev_ckpt_path}!")
else:
    print(f"[Error] Checkpoint not found at {prev_ckpt_path}.")

ema = ExponentialMovingAverage(model, decay=0.9999)


[Resume] Successfully loaded Epoch 10 Checkpoint from /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_sota_best.pt!


In [5]:
# Cell 5: AMP Training & Evaluation Helpers (Grad Accum = 4)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

def train_epoch_exp08(model, loader, criterion, optimizer, ema, device, collator=None, grad_accum_steps=4):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    for i, (images, labels) in enumerate(loader):
        if collator is not None:
            images, targets = collator(list(zip(images, labels)))
            images, targets = images.to(device), targets.to(device)
        else:
            images = images.to(device)
            targets = F.one_hot(labels, 10).float().to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, targets) / grad_accum_steps

        scaler.scale(loss).backward()
        if (i + 1) % grad_accum_steps == 0 or (i + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if ema is not None:
                ema.update()

        running_loss += loss.item() * grad_accum_steps * images.size(0)
        _, predicted = outputs.max(1)
        _, target_class = targets.max(1)
        total += labels.size(0)
        correct += predicted.eq(target_class).sum().item()

    return running_loss / total, 100.0 * correct / total

@torch.inference_mode()
def evaluate_exp08(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total

print("[Training] AMP & Gradient Accumulation (4 steps) configured.")


[Training] AMP & Gradient Accumulation (4 steps) configured.


In [6]:
# Cell 6: Resume Fine-Tuning Phase 2 (Epochs 11 to 20)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

extra_epochs = 10
start_epoch = 11
end_epoch = start_epoch + extra_epochs - 1

optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=extra_epochs, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()

best_val_acc = 98.82  # Baseline from Epoch 10
print(f"=== STARTING EXP-08 PHASE 2: RESUME FINE-TUNING (Epochs {start_epoch} to {end_epoch}) ===")

for epoch_idx in range(1, extra_epochs + 1):
    current_epoch = start_epoch + epoch_idx - 1
    t0 = time.time()
    train_loss, train_acc = train_epoch_exp08(model, loader=train_loader, criterion=criterion, optimizer=optimizer, ema=ema, device=device, collator=mixup_collator, grad_accum_steps=4)
    scheduler.step()
    
    # Evaluate EMA weights
    ema.apply_shadow()
    ema_val_loss, ema_val_acc = evaluate_exp08(model, val_loader, device)
    ema.restore()
    
    elapsed = time.time() - t0
    print(f"Epoch {current_epoch:2d}/{end_epoch:2d} [{elapsed:.1f}s] | Train Loss: {train_loss:.4f} | EMA Val Acc: {ema_val_acc:.2f}%")
    
    if ema_val_acc > best_val_acc:
        best_val_acc = ema_val_acc
        ema.apply_shadow()
        torch.save(model.state_dict(), new_ckpt_path)
        ema.restore()
        print(f"  --> 🏆 NEW ULTIMATE PEAK RECORD ({best_val_acc:.2f}%) Saved to {new_ckpt_path}!")


=== STARTING EXP-08 PHASE 2: RESUME FINE-TUNING (Epochs 11 to 20) ===
Epoch 11/20 [690.3s] | Train Loss: 0.5424 | EMA Val Acc: 98.84%
  --> 🏆 NEW ULTIMATE PEAK RECORD (98.84%) Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_20epochs_peak.pt!
Epoch 12/20 [689.9s] | Train Loss: 0.5456 | EMA Val Acc: 98.88%
  --> 🏆 NEW ULTIMATE PEAK RECORD (98.88%) Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_20epochs_peak.pt!
Epoch 13/20 [690.1s] | Train Loss: 0.5367 | EMA Val Acc: 98.94%
  --> 🏆 NEW ULTIMATE PEAK RECORD (98.94%) Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_20epochs_peak.pt!
Epoch 14/20 [690.3s] | Train Loss: 0.5200 | EMA Val Acc: 98.96%
  --> 🏆 NEW ULTIMATE PEAK RECORD (98.96%) Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_20epochs_peak.pt!
Epoch 15/20 [690.0s] | Train Loss: 0.5172 | EM

In [7]:
# Cell 7: Final Test Set Evaluation (10,000 samples)
target_ckpt = new_ckpt_path if os.path.exists(new_ckpt_path) else prev_ckpt_path
print(f"[Test Evaluation] Loading model from: {target_ckpt}")
model_best = build_model_sota("convnext_small", num_classes=10, device=device)
model_best.load_state_dict(torch.load(target_ckpt, map_location=device))

test_loss, test_acc = evaluate_exp08(model_best, test_loader, device)
print("="*65)
print(f"EXP-08 PHASE 2 FINAL TEST SET ACCURACY: 👑 {test_acc:.2f}%")
print("="*65)


[Test Evaluation] Loading model from: /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_20epochs_peak.pt
EXP-08 PHASE 2 FINAL TEST SET ACCURACY: 👑 98.87%
